In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re  # regex to clean up newlines
import hashlib  # build a stable ID (fingerprint) from text
from pathlib import Path  # get file name from path

from langchain_community.document_loaders import PyPDFDirectoryLoader  # loads every PDF in a folder
from langchain_text_splitters import RecursiveCharacterTextSplitter  # splits text into small chunks
from langchain_ollama import OllamaEmbeddings, ChatOllama  # Ollama embedding and chat model
from langchain_chroma import Chroma  # vector database

# ---------- 1. Settings (one place, easy to change) ----------
CV_FOLDER = "data/cvs"  # path to the CV folder
CHUNK_SIZE = 800  # max ~800 characters per chunk
CHUNK_OVERLAP = 100  # ~100 character overlap with the neighboring chunk
COLLECTION = "cv_chunks"  # vector store collection name
EMBED_MODEL = "nomic-embed-text"  # model used for embeddings
CHAT_MODEL = "llama3.2:3b"  # model used to answer questions

/var/folders/fx/n2sq58254dsdkq7q30ldz5qw0000gn/T/ipykernel_5221/2486048563.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader  # loads every PDF in a folder
/opt/homebrew/Caskroom/miniconda/base/envs/langchain/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ---------- 2. Load PDFs ----------
docs = PyPDFDirectoryLoader(CV_FOLDER, glob="**/*.pdf").load()  # one Document per page of each PDF

# ---------- 3. Clean text and add metadata ----------
for d in docs:  # for every page
    d.page_content = re.sub(r"\s+", " ", d.page_content).strip()  # collapse all newlines/extra spaces into one space
    d.metadata["candidate"] = Path(d.metadata["source"]).stem  # file name without extension

# ---------- 4. Split ----------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,  # max chunk size
    chunk_overlap=CHUNK_OVERLAP,  # overlap
    is_separator_regex=True,  # the separators below are regex
    separators=[
        r"\s+(?=●)",  # split before ● (bullet stays with the text after it)
        r"(?<=[.!?])\s+",  # split at end of sentence
        r" ",  # split at word boundaries
        r"",  # last resort: split by character
    ],
)
chunks = splitter.split_documents(docs)  # Document → small chunks

# ---------- 5. Attach candidate name to every chunk ----------
for c in chunks:
    c.page_content = f"CV of {c.metadata['candidate']}:\n{c.page_content}"  # so the LLM knows whose CV this is

In [4]:
# ---------- 6. Remove duplicate chunks ----------
seen = set()  # text already seen
unique_chunks = []  # chunks without duplicates
for c in chunks:
    if c.page_content not in seen:  # if it's new text
        seen.add(c.page_content)  # mark it as seen
        unique_chunks.append(c)  # add it to the list

ids = [hashlib.md5(c.page_content.encode("utf-8")).hexdigest() for c in unique_chunks]  # stable ID from text

print(f"pages: {len(docs)} | chunks: {len(chunks)} | unique chunks: {len(unique_chunks)}")  # sanity check the counts

pages: 3 | chunks: 11 | unique chunks: 11


In [5]:
# ---------- 7. Embeddings and vector store (rebuilt every run) ----------
embeddings = OllamaEmbeddings(model=EMBED_MODEL)  # model that turns text into vectors

old_store = Chroma(collection_name=COLLECTION, embedding_function=embeddings)  # handle to the previous collection
old_store.delete_collection()  # wipe the old collection so no stale copies remain

vectorstore = Chroma.from_documents(
    unique_chunks,  # chunks without duplicates
    embeddings,  # embedding model
    ids=ids,  # stable IDs so the same text is never stored twice
    collection_name=COLLECTION,  # new collection
)

print("store count:", vectorstore._collection.count(), "| expected:", len(unique_chunks))  # these two should match

store count: 11 | expected: 11


In [6]:
# ---------- 8. Search test ----------
def search(question, k=4):  # runs a query and prints the closest chunks
    print(f"\nQ: {question}")  # print the question
    for doc, score in vectorstore.similarity_search_with_score(question, k=k):  # lower score = closer match
        print(f"  {score:.3f} | {doc.metadata['candidate']} | {doc.page_content[:90]!r}")  # score, name, start of text

search("Who has experience with Image processing ?")  # example 1
search("Who worked on RAG or LangChain?")  # example 2


Q: Who has experience with Image processing ?
  0.813 | Ashikujjaman Shaikat Resume | 'CV of Ashikujjaman Shaikat Resume:\nLanguage Proficiency: - English — Fluent - Bengali — Na'
  0.888 | Ashikujjaman Shaikat Resume | 'CV of Ashikujjaman Shaikat Resume:\n- Backend Collaboration: Collaborate with backend engin'
  0.900 | Ashikujjaman Shaikat Resume | 'CV of Ashikujjaman Shaikat Resume:\nMd Ashikujjaman Shaikat PROFESSIONAL SUMMARY Senior Sof'
  0.905 | Ashikujjaman Shaikat Resume | 'CV of Ashikujjaman Shaikat Resume:\nEDUCATION Daffodil International University B.Sc. in CO'

Q: Who worked on RAG or LangChain?
  1.063 | Ashikujjaman Shaikat Resume | 'CV of Ashikujjaman Shaikat Resume:\n- Backend Collaboration: Collaborate with backend engin'
  1.094 | Ashikujjaman Shaikat Resume | 'CV of Ashikujjaman Shaikat Resume:\n- API Integration: Integrate RESTful APIs with React an'
  1.117 | Ashikujjaman Shaikat Resume | 'CV of Ashikujjaman Shaikat Resume:\n- Created shared UI utilities to imp

In [7]:
# ---------- 9. Q&A: ask the LLM to answer using the relevant chunks ----------
llm = ChatOllama(model=CHAT_MODEL, temperature=0)  # temperature=0: less random, more reliable answers

def ask(question, k=4):  # question → answer
    hits = vectorstore.similarity_search(question, k=k)  # find the closest chunks
    context = "\n\n".join(h.page_content for h in hits)  # join chunks into a context block
    prompt = (
        "Answer the question using ONLY the CV excerpts below. "  # only answer from the CVs
        "Mention the candidate name. "  # name the candidate
        "If the answer is not in the excerpts, say 'Not found in the CVs.'\n\n"  # admit if it's not found
        f"CV excerpts:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return llm.invoke(prompt).content  # return the model's answer

In [8]:
print(ask("Who has experience with Langchain?"))  # example

Not found in the CVs.
